In [ ]:
from pathlib import Path
import io
import re

import fitz
from PIL import Image
import pytesseract

import chromadb
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

c:\Users\shafw\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
BASE_DIR = Path(r"D:\code\nlp\ta-sisdas")

DOMAIN = "academic_administration"
DOMAIN_DIR = BASE_DIR / "dataset_raw" / DOMAIN
CHROMA_DIR = BASE_DIR / "chroma_db"

COLLECTION_NAME = "academic_administration"
EMBEDDING_MODEL = "bge-m3"

RESET_COLLECTION = True

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

NameError: name 'Path' is not defined

In [3]:
DOCUMENT_METADATA_MAP = {
    "Kalender-Akademik-2026-2027.pdf": {
        "document_type": "academic_calendar",
        "academic_year": "2026/2027",
        "topic": "academic_calendar"
    },

    "KALENDER-AKADEMIK-UNIVERSITAS-NEGERI-MALANG-TAHUN-AKADEMIK-2024-2025.pdf": {
        "document_type": "academic_calendar",
        "academic_year": "2024/2025",
        "topic": "academic_calendar"
    },

    "Pedoman-Pendidikan-Edisi-2020_Final.pdf": {
        "document_type": "academic_guide",
        "academic_year": "general",
        "topic": "pedoman_pendidikan"
    },

    "PER-NO-12-TAHUN-2018-PEDOMAN-PENDIDIKAN-UNIVERSITAS-NEGERI-MALANG-TAHUN-AKADEMIK-2018-2019.pdf": {
        "document_type": "academic_guide",
        "academic_year": "general",
        "topic": "pedoman_pendidikan"
    },

    "PER-NO-18-TAHUN-2018-PENYELENGGARAAN-PENDIDIKAN-UNIVERSITAS-NEGERI-MALANG.pdf": {
        "document_type": "academic_regulation",
        "academic_year": "general",
        "topic": "general"
    },

    "PER-NO-19-TAHUN-2018-TAHUN-AKADEMIK-KALENDER-AKADEMIK-DAN-SISTEM-ADMINISTRASI-AKADEMIK-UNIVERSITAS-NEGERI-MALANG.pdf": {
        "document_type": "academic_regulation",
        "academic_year": "general",
        "topic": "administrasi_akademik"
    },

    "PER-NO-20-TAHUN-2018-KURIKULUM-UNIVERSITAS-NEGERI-MALANG.pdf": {
        "document_type": "academic_regulation",
        "academic_year": "general",
        "topic": "kurikulum"
    },

    "PER-NO-21-TAHUN-2018-PENILAIAN-DAN-PROSES-HASIL-BELAJAR-MAHASISWA-UNIVERSITAS-NEGERI-MALANG.pdf": {
        "document_type": "academic_regulation",
        "academic_year": "general",
        "topic": "penilaian_hasil_belajar"
    },

    "PER-NO-28-TAHUN-2018-KEBEBASAN-AKADEMIK-KEBEBASAN-MIMBAR-AKADEMIK-DANOTONOMI-KEILMUAN-UNIVERSITAS-NEGERI-MALANG.pdf": {
        "document_type": "academic_regulation",
        "academic_year": "general",
        "topic": "general"
    },

    "Perubahan-Pedoman-Akademik-Edisi-2020-UM-.pdf": {
        "document_type": "academic_regulation",
        "academic_year": "general",
        "topic": "pedoman_pendidikan"
    }
}

def infer_document_metadata(pdf_path: Path) -> dict:
    file_name = pdf_path.name

    if file_name not in DOCUMENT_METADATA_MAP:
        raise ValueError(
            f"Metadata untuk file ini belum diset manual: {file_name}"
        )

    return DOCUMENT_METADATA_MAP[file_name]

In [4]:
def make_safe_id(text: str) -> str:
    text = text.replace(" ", "_")
    text = re.sub(r"[^a-zA-Z0-9_\-]", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")


def load_pdf_normal(pdf_path: Path, domain: str):
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()

    cleaned_docs = []

    for doc in docs:
        text = doc.page_content.strip()

        if not text:
            continue

        # PyPDFLoader biasanya page mulai dari 0, kita ubah jadi 1-based
        page = doc.metadata.get("page", 0)
        try:
            page = int(page) + 1
        except Exception:
            page = None
        extra_metadata = infer_document_metadata(pdf_path)
        doc.metadata = {
            "domain": domain,
            "source": str(pdf_path),
            "file_name": pdf_path.name,
            "page": page,
            "extraction_method": "pypdf",
            **extra_metadata
        }

        cleaned_docs.append(doc)

    return cleaned_docs


def load_pdf_ocr(pdf_path: Path, domain: str):
    pdf = fitz.open(str(pdf_path))
    docs = []

    for page_number, page in enumerate(pdf, start=1):
        pix = page.get_pixmap(matrix=fitz.Matrix(3, 3), alpha=False)
        image = Image.open(io.BytesIO(pix.tobytes("png")))

        text = pytesseract.image_to_string(image, lang="ind+eng").strip()

        print(f"  OCR page {page_number}: {len(text)} chars")

        if text:
            extra_metadata = infer_document_metadata(pdf_path)
            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "domain": domain,
                        "source": str(pdf_path),
                        "file_name": pdf_path.name,
                        "page": page_number,
                        "extraction_method": "ocr_tesseract_ind_eng",
                        **extra_metadata
                    }
                )
            )

    pdf.close()
    return docs


def load_pdf_smart(pdf_path: Path, domain: str, min_chars: int = 100):
    print(f"\nLoading: {pdf_path.name}")

    normal_docs = load_pdf_normal(pdf_path, domain)
    normal_chars = sum(len(doc.page_content) for doc in normal_docs)

    print(f"  Normal extraction chars: {normal_chars}")

    if normal_chars >= min_chars:
        print("  Using normal extraction")
        return normal_docs

    print("  Normal extraction too small. Using OCR")
    return load_pdf_ocr(pdf_path, domain)


In [ ]:
    # =====================
    # PREPARE CHROMA
    # =====================

    embeddings = OllamaEmbeddings(
        model=EMBEDDING_MODEL
    )

    test_vector = embeddings.embed_query("tes embedding")
    print("Embedding dimension:", len(test_vector))

    if len(test_vector) == 0:
        raise ValueError("Embedding gagal. Pastikan Ollama jalan dan model bge-m3 sudah di-pull.")


    client = chromadb.PersistentClient(path=str(CHROMA_DIR))

    if RESET_COLLECTION:
        try:
            client.delete_collection(name=COLLECTION_NAME)
            print(f"Old collection deleted: {COLLECTION_NAME}")
        except Exception:
            print(f"No old collection found: {COLLECTION_NAME}")


    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(CHROMA_DIR)
    )

Embedding dimension: 1024
No old collection found: academic_administration


In [6]:
# =====================
# TEXT SPLITTER
# =====================
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=100
)

In [7]:
# =====================
# INDEX ALL PDF IN DOMAIN
# =====================

pdf_files = sorted(DOMAIN_DIR.glob("*.pdf"))

print(f"\nTotal PDF files found: {len(pdf_files)}")

total_docs = 0
total_chunks = 0
failed_files = []

for pdf_path in pdf_files:
    try:
        docs = load_pdf_smart(pdf_path, DOMAIN)

        print(f"Total docs/pages loaded: {len(docs)}")

        if len(docs) == 0:
            print("  Skipped: no text extracted")
            failed_files.append((pdf_path.name, "No text extracted"))
            continue

        splits = text_splitter.split_documents(docs)

        splits = [
            split for split in splits
            if split.page_content and split.page_content.strip()
        ]

        file_key = make_safe_id(pdf_path.stem)

        for i, split in enumerate(splits):
            split.metadata["chunk_index"] = i
            split.metadata["file_key"] = file_key

        ids = [
            f"{DOMAIN}-{file_key}-page-{doc.metadata.get('page')}-chunk-{doc.metadata.get('chunk_index')}"
            for doc in splits
        ]

        print(f"  Total chunks: {len(splits)}")

        if len(splits) == 0:
            print("  Skipped: no chunks created")
            failed_files.append((pdf_path.name, "No chunks created"))
            continue

        vectorstore.add_documents(
            documents=splits,
            ids=ids
        )

        total_docs += len(docs)
        total_chunks += len(splits)

        print(f"  Indexed successfully: {pdf_path.name}")

    except Exception as e:
        print(f"  Failed: {pdf_path.name}")
        print(f"  Error: {e}")
        failed_files.append((pdf_path.name, str(e)))    


Total PDF files found: 10

Loading: Kalender-Akademik-2026-2027.pdf
  Normal extraction chars: 0
  Normal extraction too small. Using OCR
  OCR page 1: 1497 chars
  OCR page 2: 943 chars
  OCR page 3: 1324 chars
  OCR page 4: 1445 chars
  OCR page 5: 1635 chars
  OCR page 6: 1803 chars
  OCR page 7: 2023 chars
  OCR page 8: 106 chars
Total docs/pages loaded: 8
  Total chunks: 14
  Indexed successfully: Kalender-Akademik-2026-2027.pdf

Loading: KALENDER-AKADEMIK-UNIVERSITAS-NEGERI-MALANG-TAHUN-AKADEMIK-2024-2025.pdf
  Normal extraction chars: 0
  Normal extraction too small. Using OCR
  OCR page 1: 1503 chars
  OCR page 2: 918 chars
  OCR page 3: 981 chars
  OCR page 4: 1281 chars
  OCR page 5: 2036 chars
  OCR page 6: 2138 chars
  OCR page 7: 1251 chars
Total docs/pages loaded: 7
  Total chunks: 13
  Indexed successfully: KALENDER-AKADEMIK-UNIVERSITAS-NEGERI-MALANG-TAHUN-AKADEMIK-2024-2025.pdf

Loading: Pedoman-Pendidikan-Edisi-2020_Final.pdf
  Normal extraction chars: 221895
  Using 

In [8]:
# =====================
# SUMMARY
# =====================

print("\n" + "=" * 100)
print("INDEXING SUMMARY")
print("=" * 100)

print("Domain:", DOMAIN)
print("Collection:", COLLECTION_NAME)
print("Total PDFs:", len(pdf_files))
print("Total docs/pages:", total_docs)
print("Total chunks indexed:", total_chunks)
print("Total data in collection:", vectorstore._collection.count())

if failed_files:
    print("\nFailed files:")
    for file_name, reason in failed_files:
        print("-", file_name, "=>", reason)
else:
    print("\nNo failed files.")


INDEXING SUMMARY
Domain: academic_administration
Collection: academic_administration
Total PDFs: 10
Total docs/pages: 431
Total chunks indexed: 835
Total data in collection: 835

No failed files.


In [9]:
import chromadb
from pathlib import Path
from collections import Counter

CHROMA_DIR = Path(r"D:\code\nlp\ta-sisdas\chroma_db")

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(name="academic_administration")

print("Total chunks:", collection.count())

data = collection.get(
    include=["metadatas"]
)

counter = Counter(
    metadata.get("file_name", "UNKNOWN")
    for metadata in data["metadatas"]
)

for file_name, count in counter.items():
    print(file_name, ":", count, "chunks")

Total chunks: 835
Kalender-Akademik-2026-2027.pdf : 14 chunks
KALENDER-AKADEMIK-UNIVERSITAS-NEGERI-MALANG-TAHUN-AKADEMIK-2024-2025.pdf : 13 chunks
Pedoman-Pendidikan-Edisi-2020_Final.pdf : 298 chunks
PER-NO-12-TAHUN-2018-PEDOMAN-PENDIDIKAN-UNIVERSITAS-NEGERI-MALANG-TAHUN-AKADEMIK-2018-2019.pdf : 272 chunks
PER-NO-18-TAHUN-2018-PENYELENGGARAAN-PENDIDIKAN-UNIVERSITAS-NEGERI-MALANG.pdf : 56 chunks
PER-NO-19-TAHUN-2018-TAHUN-AKADEMIK-KALENDER-AKADEMIK-DAN-SISTEM-ADMINISTRASI-AKADEMIK-UNIVERSITAS-NEGERI-MALANG.pdf : 38 chunks
PER-NO-20-TAHUN-2018-KURIKULUM-UNIVERSITAS-NEGERI-MALANG.pdf : 67 chunks
PER-NO-21-TAHUN-2018-PENILAIAN-DAN-PROSES-HASIL-BELAJAR-MAHASISWA-UNIVERSITAS-NEGERI-MALANG.pdf : 48 chunks
PER-NO-28-TAHUN-2018-KEBEBASAN-AKADEMIK-KEBEBASAN-MIMBAR-AKADEMIK-DANOTONOMI-KEILMUAN-UNIVERSITAS-NEGERI-MALANG.pdf : 7 chunks
Perubahan-Pedoman-Akademik-Edisi-2020-UM-.pdf : 22 chunks


In [12]:
from collections import defaultdict
import chromadb
from pathlib import Path

CHROMA_DIR = Path(r"D:\code\nlp\ta-sisdas\chroma_db")
COLLECTION_NAME = "academic_administration"

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_collection(name=COLLECTION_NAME)

data = collection.get(include=["metadatas"])

metadata_values = defaultdict(set)

for metadata in data["metadatas"]:
    for key, value in metadata.items():
        metadata_values[key].add(value)

for key, values in metadata_values.items():
    print("=" * 80)
    print("Metadata key:", key)
    print("Values:")
    for value in sorted(values, key=lambda x: str(x)):
        print("-", value)

Metadata key: file_key
Values:
- KALENDER-AKADEMIK-UNIVERSITAS-NEGERI-MALANG-TAHUN-AKADEMIK-2024-2025
- Kalender-Akademik-2026-2027
- PER-NO-12-TAHUN-2018-PEDOMAN-PENDIDIKAN-UNIVERSITAS-NEGERI-MALANG-TAHUN-AKADEMIK-2018-2019
- PER-NO-18-TAHUN-2018-PENYELENGGARAAN-PENDIDIKAN-UNIVERSITAS-NEGERI-MALANG
- PER-NO-19-TAHUN-2018-TAHUN-AKADEMIK-KALENDER-AKADEMIK-DAN-SISTEM-ADMINISTRASI-AKADEMIK-UNIVERSITAS-NEGERI-MALANG
- PER-NO-20-TAHUN-2018-KURIKULUM-UNIVERSITAS-NEGERI-MALANG
- PER-NO-21-TAHUN-2018-PENILAIAN-DAN-PROSES-HASIL-BELAJAR-MAHASISWA-UNIVERSITAS-NEGERI-MALANG
- PER-NO-28-TAHUN-2018-KEBEBASAN-AKADEMIK-KEBEBASAN-MIMBAR-AKADEMIK-DANOTONOMI-KEILMUAN-UNIVERSITAS-NEGERI-MALANG
- Pedoman-Pendidikan-Edisi-2020_Final
- Perubahan-Pedoman-Akademik-Edisi-2020-UM-
Metadata key: academic_year
Values:
- 2024/2025
- 2026/2027
- general
Metadata key: file_name
Values:
- KALENDER-AKADEMIK-UNIVERSITAS-NEGERI-MALANG-TAHUN-AKADEMIK-2024-2025.pdf
- Kalender-Akademik-2026-2027.pdf
- PER-NO-12-TAHUN-2018-P